# Dataset 1 NMC — FULL model-pool runtime ablation

This notebook runs the complete manuscript tabular model pool, Kadem-style GPR-EKF, best single estimator, full MAW-GME, and Fast-MAW-GME K3/K5 under the same Google Colab runtime protocol.


In [ ]:

# ============================================================
# FULL MODEL-POOL MAW-GME RUNTIME ABLATION + KADEM-STYLE BASELINE
# Google Colab-ready, dataset-specific notebook with all manuscript tabular estimators
# ============================================================

# -----------------------------
# 0) Google Drive mount
# -----------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped or unavailable:', e)

# -----------------------------
# 1) Install/import dependencies
# -----------------------------
import os, sys, time, math, pickle, warnings, subprocess, importlib.util
# Prevent Colab/BLAS thread oversubscription during many small model fits.
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
from pathlib import Path
warnings.filterwarnings('ignore')

def pip_install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Core packages
for pkg, imp in [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('scipy', 'scipy'),
    ('scikit-learn', 'sklearn'),
    ('openpyxl', 'openpyxl'),
    ('tqdm', 'tqdm'),
]:
    pip_install_if_missing(pkg, imp)

# Optional boosting packages used in the manuscript model pool.
# These are installed here so that the runtime comparison is complete rather than lightweight-only.
for pkg, imp in [
    ('xgboost', 'xgboost'),
    ('lightgbm', 'lightgbm'),
    ('catboost', 'catboost'),
]:
    pip_install_if_missing(pkg, imp)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception as e:
    XGBRegressor = None
    XGB_AVAILABLE = False
    print('XGBoost unavailable:', e)
try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except Exception as e:
    LGBMRegressor = None
    LGBM_AVAILABLE = False
    print('LightGBM unavailable:', e)
try:
    from catboost import CatBoostRegressor
    CAT_AVAILABLE = True
except Exception as e:
    CatBoostRegressor = None
    CAT_AVAILABLE = False
    print('CatBoost unavailable:', e)




# ============================================================
# 2) DATASET-SPECIFIC CONFIGURATION
#    Paths are copied from the previous MAW-GME notebooks.
# ============================================================
DATASET_LABEL = 'Dataset 1'
DRIVE_ROOT = '/content/drive/MyDrive/Batarya'
FEATURE_CSV = '/content/drive/MyDrive/Batarya/Dataset_1/Dataset1_NMC_MAW_GME_results/features/multi_hi_features.csv'
RESULT_DIR = '/content/drive/MyDrive/Batarya/Dataset_1/Dataset1_NMC_MAW_GME_results/results'
WINDOWS = {'full_32_42': 'Full 3.20-4.20 V', 'medium_35_40': 'Medium 3.50-4.00 V', 'narrow_36_385': 'Narrow 3.60-3.85 V', 'very_narrow_365_38': 'Very narrow 3.65-3.80 V'}
KADEM_PMAX_WINDOW = 'narrow_36_385'
PAPER_GPR_EKF_RMSE = 0.0188
PAPER_GPR_EKF_MAE = 0.0156

# Runtime/accuracy ablation settings.
RANDOM_STATE = 42
TOP_KS = [3, 5]
LATENCY_AWARE_PRUNING = True
SIZE_PENALTY = 0.02
# 'fast_holdout' is much faster for Colab runtime ablation.
# Change to 'inner_loco' for a stricter but slower nested selection.
INNER_SELECTION_MODE = 'inner_loco'
BMS_MIN_AVAILABILITY = 0.50
BMS_AVAILABILITY_GAMMA = 2.0
BMS_EPS = 1e-8

# Complete tabular model pool used in the manuscript.
# This intentionally includes the heavier boosting models for a full academic comparison.
MODEL_CANDIDATES = ['Ridge', 'ElasticNet', 'SVR', 'RandomForest', 'ExtraTrees', 'HistGB', 'XGBoost', 'LightGBM', 'CatBoost']

# The complete manuscript pool below uses the same tree/boosting settings as the original full notebooks.
# These legacy variables are kept only for compatibility with older cells; they are not used by make_tabular_models().
N_TREES_FAST = 80
MAX_DEPTH_FAST = 6
MIN_SAMPLES_LEAF_FAST = 2

FAST_OUT_DIR = os.path.join(RESULT_DIR, 'bms_online_results', 'full_model_pool_runtime_review')
os.makedirs(FAST_OUT_DIR, exist_ok=True)

print('DATASET_LABEL:', DATASET_LABEL)
print('FEATURE_CSV:', FEATURE_CSV)
print('FEATURE_CSV exists:', os.path.exists(FEATURE_CSV))
print('RESULT_DIR:', RESULT_DIR)
print('FAST_OUT_DIR:', FAST_OUT_DIR)
print('WINDOWS:', WINDOWS)
print('KADEM_PMAX_WINDOW:', KADEM_PMAX_WINDOW)

# ============================================================
# 3) Load and standardize the feature table
# ============================================================
if not os.path.exists(FEATURE_CSV):
    raise FileNotFoundError(
        'Feature CSV not found. First run the original feature-extraction notebook, or update FEATURE_CSV.\n'
        f'Expected path: {FEATURE_CSV}'
    )

df = pd.read_csv(FEATURE_CSV)
if 'cell' not in df.columns or 'soh' not in df.columns:
    raise ValueError('Feature CSV must contain at least cell and soh columns.')

# Use cycle_number as chronological BMS update index. Fall back to row order if missing.
if 'cycle_number' not in df.columns:
    df['cycle_number'] = df.groupby('cell').cumcount().astype(float)

df['cell'] = df['cell'].astype(str)
df['soh'] = pd.to_numeric(df['soh'], errors='coerce')
df['cycle_number'] = pd.to_numeric(df['cycle_number'], errors='coerce')
df = df.dropna(subset=['cell', 'soh']).copy()
df = df.sort_values(['cell', 'cycle_number']).reset_index(drop=True)

print('Loaded feature table:', df.shape)
print('Cells:', sorted(df['cell'].unique()))
display(df.groupby('cell').agg(n_rows=('soh','size'), first_cycle=('cycle_number','min'), last_cycle=('cycle_number','max'), first_soh=('soh','first'), last_soh=('soh','last')).reset_index())

# ============================================================
# 4) Utility functions
# ============================================================
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask])))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan
    return float(mean_absolute_error(y_true[mask], y_pred[mask]))

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    out = {
        'rmse': np.nan,
        'mae': np.nan,
        'r2': np.nan,
        'n_eval': int(mask.sum()),
    }
    if mask.sum() > 0:
        out['rmse'] = float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask])))
        out['mae'] = float(mean_absolute_error(y_true[mask], y_pred[mask]))
        out['r2'] = float(r2_score(y_true[mask], y_pred[mask])) if mask.sum() >= 2 else np.nan
    return out

def safe_model_size_kb(obj):
    try:
        return float(len(pickle.dumps(obj)) / 1024.0)
    except Exception:
        return np.nan

def make_tabular_models():
    """Complete Dataset-1 tabular model pool used in the manuscript.
    This is not the lightweight subset. It includes Ridge, ElasticNet, SVR,
    RandomForest, ExtraTrees, HistGB, XGBoost, LightGBM, and CatBoost.
    """
    models = {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        ]),
        "ElasticNet": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.0005, l1_ratio=0.2, max_iter=20000, random_state=RANDOM_STATE)),
        ]),
        "SVR": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf", C=20.0, gamma="scale", epsilon=0.002)),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=1, n_jobs=-1)),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=1, n_jobs=-1)),
        ]),
        "HistGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(max_iter=250, learning_rate=0.04, random_state=RANDOM_STATE)),
        ]),
    }
    if XGB_AVAILABLE:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=350, learning_rate=0.03, max_depth=3,
                subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE,
                objective="reg:squarederror", n_jobs=-1
            )),
        ])
    if LGBM_AVAILABLE:
        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)),
        ])
    if CAT_AVAILABLE:
        models["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(iterations=300, learning_rate=0.03, depth=4, random_seed=RANDOM_STATE, verbose=False)),
        ])
    return models

# Verify that the complete manuscript model pool is available after make_tabular_models() is defined.
print('Requested complete MODEL_CANDIDATES:', MODEL_CANDIDATES)
_available_probe = make_tabular_models()
missing_requested_models = [m for m in MODEL_CANDIDATES if m not in _available_probe]
print('Available model pool:', list(_available_probe.keys()))
if missing_requested_models:
    raise RuntimeError('Some requested manuscript models are unavailable after installation/import: ' + str(missing_requested_models))


TARGET_DERIVED_EXCLUDE = {
    'soh', 'capacity_mAh', 'capacity_Ah', 'soh_rolling_median', 'soh_delta', 'soh_outlier',
    'cycle_name', 'source_file', 'rpt_group', 'rpt_index'
}

def get_window_feature_cols(dataframe, window_name, use_cycle_number=True, use_global_temperature=True):
    cols = []
    for c in dataframe.columns:
        if c in TARGET_DERIVED_EXCLUDE:
            continue
        if c == 'cell':
            continue
        if c == 'cycle_number':
            continue
        if c.endswith('_' + window_name):
            cols.append(c)
    if use_cycle_number and 'cycle_number' in dataframe.columns:
        cols = ['cycle_number'] + cols
    if use_global_temperature:
        for c in ['temp_mean', 'temp_max', 'temp_min', 'temp_std', 'num_cycles_increment']:
            if c in dataframe.columns and c not in cols and c not in TARGET_DERIVED_EXCLUDE:
                cols.append(c)
    # Keep numeric columns only.
    cols = [c for c in cols if c in dataframe.columns and pd.api.types.is_numeric_dtype(dataframe[c])]
    if len(cols) == 0:
        raise ValueError(f'No features found for window={window_name}')
    return cols

def feature_availability(row, cols):
    vals = pd.to_numeric(row.reindex(cols), errors='coerce').to_numpy(dtype=float)
    return float(np.isfinite(vals).mean()) if len(vals) else 0.0

# ============================================================
# 5) Inner-LOCO candidate ranking and deployment model fitting
# ============================================================
def inner_loco_rank_candidates(train_df, selected_models=None, latency_aware=False, size_penalty=0.02):
    selected_models = selected_models or MODEL_CANDIDATES
    base_models = make_tabular_models()
    selected_models = [m for m in selected_models if m in base_models]
    train_cells = sorted(train_df['cell'].unique())
    rows = []

    for window_name in WINDOWS.keys():
        try:
            cols = get_window_feature_cols(train_df, window_name, True, True)
        except Exception as e:
            print('Skip window:', window_name, e)
            continue
        for model_name in selected_models:
            fold_rmses = []
            fold_maes = []
            valid_folds = 0
            size_values = []
            # fast_holdout keeps the reviewer ablation practical in Colab; inner_loco is stricter but slower.
            validation_cells = [train_cells[-1]] if INNER_SELECTION_MODE == 'fast_holdout' else train_cells
            for val_cell in validation_cells:
                fit_df = train_df[train_df['cell'] != val_cell].copy()
                val_df = train_df[train_df['cell'] == val_cell].copy()
                val_df = val_df[val_df[cols].notna().mean(axis=1) >= BMS_MIN_AVAILABILITY].copy()
                if len(fit_df) < 5 or len(val_df) < 2:
                    continue
                try:
                    model = clone(base_models[model_name])
                    model.fit(fit_df[cols], fit_df['soh'].values.astype(float))
                    pred = np.clip(model.predict(val_df[cols]), 0.5, 1.05)
                    fold_rmses.append(rmse(val_df['soh'].values, pred))
                    fold_maes.append(mae(val_df['soh'].values, pred))
                    size_values.append(safe_model_size_kb(model))
                    valid_folds += 1
                except Exception as e:
                    print(f'Candidate failed: window={window_name}, model={model_name}, val_cell={val_cell}: {e}')
            if valid_folds == 0:
                continue
            mean_rmse = float(np.nanmean(fold_rmses))
            mean_mae = float(np.nanmean(fold_maes))
            mean_size = float(np.nanmean(size_values)) if len(size_values) else np.nan
            rank_score = mean_rmse
            if latency_aware:
                rank_score = mean_rmse * (1.0 + size_penalty * np.log1p(max(mean_size, 0.0)))
            rows.append({
                'window': window_name,
                'model_name': model_name,
                'feature_cols': cols,
                'inner_rmse': mean_rmse,
                'inner_mae': mean_mae,
                'inner_valid_folds': int(valid_folds),
                'inner_model_size_kb': mean_size,
                'rank_score': float(rank_score),
                'n_features': int(len(cols)),
            })
    if not rows:
        raise RuntimeError('No valid candidate experts were found during inner-LOCO ranking.')
    cand = pd.DataFrame(rows).sort_values('rank_score', ascending=True).reset_index(drop=True)
    return cand

def fit_direct_deploy(train_df, candidate_row):
    base_models = make_tabular_models()
    cols = list(candidate_row['feature_cols'])
    model_name = candidate_row['model_name']
    model = clone(base_models[model_name])
    model.fit(train_df[cols], train_df['soh'].values.astype(float))
    return {
        'kind': 'Best-single-estimator',
        'model_name': model_name,
        'window': candidate_row['window'],
        'feature_cols': cols,
        'model': model,
        'val_rmse': float(candidate_row['inner_rmse']),
        'rank_score': float(candidate_row['rank_score']),
        'model_size_kb': safe_model_size_kb(model),
        'n_experts': 1,
    }

def fit_maw_gme_deploy(train_df, candidate_ranking, top_k=None, latency_label=False):
    base_models = make_tabular_models()
    if top_k is None:
        selected = candidate_ranking.copy()
        label = f'MAW-GME-full-{len(selected)}experts'
    else:
        selected = candidate_ranking.head(int(top_k)).copy()
        label = f'Fast-MAW-GME-K{len(selected)}'
        if latency_label:
            label += '-latency-aware'
    experts = []
    for _, r in selected.iterrows():
        cols = list(r['feature_cols'])
        model_name = r['model_name']
        model = clone(base_models[model_name])
        model.fit(train_df[cols], train_df['soh'].values.astype(float))
        size_kb = safe_model_size_kb(model)
        weight = 1.0 / (float(r['rank_score']) + BMS_EPS)
        experts.append({
            'window': r['window'],
            'model_name': model_name,
            'feature_cols': cols,
            'model': model,
            'inner_rmse': float(r['inner_rmse']),
            'rank_score': float(r['rank_score']),
            'base_weight': float(weight),
            'model_size_kb': float(size_kb),
            'n_features': int(len(cols)),
        })
    total_w = sum(e['base_weight'] for e in experts)
    for e in experts:
        e['base_weight_norm'] = float(e['base_weight'] / total_w) if total_w > 0 else 1.0 / len(experts)
    val_rmse = float(np.sum([e['inner_rmse'] * e['base_weight_norm'] for e in experts]))
    return {
        'kind': label,
        'model_name': label,
        'window': 'adaptive_windows',
        'experts': experts,
        'val_rmse': val_rmse,
        'rank_score': val_rmse,
        'model_size_kb': float(np.nansum([e['model_size_kb'] for e in experts])),
        'n_experts': int(len(experts)),
        'candidate_ranking': candidate_ranking.drop(columns=['feature_cols'], errors='ignore').copy(),
    }

def predict_direct_one(deploy, row):
    t0 = time.perf_counter()
    cols = deploy['feature_cols']
    availability = feature_availability(row, cols)
    feature_ms = (time.perf_counter() - t0) * 1000.0
    if availability < BMS_MIN_AVAILABILITY:
        return np.nan, availability, feature_ms, 0.0, 'no_measurement'
    t1 = time.perf_counter()
    X_one = pd.DataFrame([pd.to_numeric(row.reindex(cols), errors='coerce').to_numpy(dtype=float)], columns=cols)
    pred = float(np.clip(deploy['model'].predict(X_one)[0], 0.5, 1.05))
    infer_ms = (time.perf_counter() - t1) * 1000.0
    return pred, availability, feature_ms, infer_ms, 'measurement_update'

def predict_maw_one(deploy, row, allowed_windows=None):
    allowed_windows = None if allowed_windows is None else set(allowed_windows)
    t0 = time.perf_counter()
    pred_items = []
    for e in deploy['experts']:
        if allowed_windows is not None and e['window'] not in allowed_windows:
            continue
        cols = e['feature_cols']
        vals = pd.to_numeric(row.reindex(cols), errors='coerce').to_numpy(dtype=float)
        availability = float(np.isfinite(vals).mean()) if len(vals) else 0.0
        if availability < BMS_MIN_AVAILABILITY:
            continue
        dynamic_weight = e['base_weight_norm'] * (availability ** BMS_AVAILABILITY_GAMMA)
        pred_items.append((e, vals, availability, dynamic_weight))
    feature_ms = (time.perf_counter() - t0) * 1000.0
    if not pred_items:
        return np.nan, 0.0, feature_ms, 0.0, 'no_measurement'
    t1 = time.perf_counter()
    preds, weights, avails = [], [], []
    for e, vals, availability, w in pred_items:
        cols = e['feature_cols']
        X_one = pd.DataFrame([vals], columns=cols)
        try:
            p = float(np.clip(e['model'].predict(X_one)[0], 0.5, 1.05))
            preds.append(p)
            weights.append(w)
            avails.append(availability)
        except Exception as ex:
            print('Prediction failed:', e['window'], e['model_name'], ex)
    infer_ms = (time.perf_counter() - t1) * 1000.0
    if len(preds) == 0 or np.sum(weights) <= 0:
        return np.nan, 0.0, feature_ms, infer_ms, 'no_measurement'
    weights = np.asarray(weights, dtype=float)
    weights /= weights.sum()
    pred = float(np.sum(np.asarray(preds, dtype=float) * weights))
    return float(np.clip(pred, 0.5, 1.05)), float(np.mean(avails)), feature_ms, infer_ms, 'measurement_update'

# ============================================================
# 6) Kadem-style GPR-EKF baseline under the same Colab environment
# ============================================================
class ScaledGPR:
    def __init__(self, n_features, random_state=42):
        self.n_features = n_features
        self.random_state = random_state
        self.x_scaler = StandardScaler()
        self.y_scaler = StandardScaler()
        kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(n_features), length_scale_bounds=(1e-3, 1e3)) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
        # optimizer=None makes the Colab baseline fast and deterministic.
        # Set optimizer='fmin_l_bfgs_b' and n_restarts_optimizer=0 for a slower tuned sklearn GPR.
        self.model = GaussianProcessRegressor(kernel=kernel, normalize_y=False, random_state=random_state, optimizer=None, alpha=1e-6)
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        Xs = self.x_scaler.fit_transform(X)
        ys = self.y_scaler.fit_transform(y).ravel()
        self.model.fit(Xs, ys)
        return self
    def predict(self, X, return_std=False):
        X = np.asarray(X, dtype=float)
        Xs = self.x_scaler.transform(X)
        if return_std:
            mu_s, std_s = self.model.predict(Xs, return_std=True)
            mu = self.y_scaler.inverse_transform(mu_s.reshape(-1, 1)).ravel()
            std = std_s * float(self.y_scaler.scale_[0])
            return mu, std
        mu_s = self.model.predict(Xs)
        return self.y_scaler.inverse_transform(mu_s.reshape(-1, 1)).ravel()

def load_kadem_single_hi_table(dataframe, window_name):
    pmax_col = 'pmax_norm_' + window_name
    if pmax_col not in dataframe.columns:
        raise ValueError(f'Kadem-style pmax column not found: {pmax_col}')
    out = dataframe[['cell', 'cycle_number', 'soh', pmax_col]].copy()
    out = out.rename(columns={pmax_col: 'pmax_norm'})
    out['cycle_index'] = out['cycle_number']
    return out.sort_values(['cell', 'cycle_index']).reset_index(drop=True)

def build_transition_training_data(train_df):
    X, y = [], []
    for _, g in train_df.sort_values(['cell', 'cycle_index']).groupby('cell'):
        g = g.reset_index(drop=True)
        for i in range(1, len(g)):
            prev_soh = float(g.loc[i-1, 'soh'])
            prev_cycle = float(g.loc[i-1, 'cycle_index'])
            cur_cycle = float(g.loc[i, 'cycle_index'])
            du = cur_cycle - prev_cycle
            if not np.isfinite(du) or du <= 0:
                du = 1.0
            X.append([prev_soh, prev_cycle, du])
            y.append(float(g.loc[i, 'soh']))
    return np.asarray(X, dtype=float), np.asarray(y, dtype=float)

def finite_jacobian_pred(pred_gpr, x, cycle, du, delta=1e-4):
    xp = np.array([[x + delta, cycle, du]], dtype=float)
    xm = np.array([[x - delta, cycle, du]], dtype=float)
    return float((pred_gpr.predict(xp)[0] - pred_gpr.predict(xm)[0]) / (2.0 * delta))

def finite_jacobian_meas(meas_gpr, x, delta=1e-4):
    xp = np.array([[x + delta]], dtype=float)
    xm = np.array([[x - delta]], dtype=float)
    return float((meas_gpr.predict(xp)[0] - meas_gpr.predict(xm)[0]) / (2.0 * delta))

def fit_kadem_gpr_ekf_models(train_df):
    Xp, yp = build_transition_training_data(train_df)
    if len(Xp) < 8:
        raise RuntimeError(f'Not enough transition samples for GPR prediction model: {len(Xp)}')
    pred_gpr = ScaledGPR(n_features=3, random_state=RANDOM_STATE).fit(Xp, yp)
    meas_df = train_df.dropna(subset=['soh', 'pmax_norm']).copy()
    if len(meas_df) < 8:
        raise RuntimeError(f'Not enough measurement samples for GPR measurement model: {len(meas_df)}')
    meas_gpr = ScaledGPR(n_features=1, random_state=RANDOM_STATE).fit(meas_df[['soh']].values, meas_df['pmax_norm'].values)
    return pred_gpr, meas_gpr

def run_kadem_gpr_ekf_on_cell(test_df, pred_gpr, meas_gpr, availability_prob=1.0, seed=42):
    rng = np.random.default_rng(seed)
    g = test_df.sort_values('cycle_index').reset_index(drop=True).copy()
    x = 1.0
    P = 0.01
    prev_cycle = float(g.loc[0, 'cycle_index']) if len(g) else 0.0
    rows = []
    for idx, row in g.iterrows():
        t0 = time.perf_counter()
        cycle = float(row['cycle_index'])
        y_true = float(row['soh'])
        z = row['pmax_norm'] if pd.notna(row['pmax_norm']) else np.nan
        if idx == 0:
            x_pred = x
            P_pred = P
        else:
            du = cycle - prev_cycle
            if not np.isfinite(du) or du <= 0:
                du = 1.0
            pred_mean, pred_std = pred_gpr.predict([[x, prev_cycle, du]], return_std=True)
            x_pred = float(np.clip(pred_mean[0], 0.5, 1.05))
            q = max(float(pred_std[0]) ** 2, 1e-7)
            F = float(finite_jacobian_pred(pred_gpr, x, prev_cycle, du))
            P_pred = F * P * F + q
        measurement_available = bool(np.isfinite(z) and (rng.random() <= availability_prob))
        if measurement_available:
            h_mean, h_std = meas_gpr.predict([[x_pred]], return_std=True)
            h_pred = float(h_mean[0])
            r = max(float(h_std[0]) ** 2, 1e-7)
            H = float(finite_jacobian_meas(meas_gpr, x_pred))
            S = H * P_pred * H + r
            K = (P_pred * H / S) if S > 0 else 0.0
            x_upd = x_pred + K * (float(z) - h_pred)
            P_upd = (1.0 - K * H) * P_pred
            mode = 'measurement_update'
        else:
            x_upd = x_pred
            P_upd = P_pred
            mode = 'prediction_only'
        x_upd = float(np.clip(x_upd, 0.5, 1.05))
        P_upd = float(max(P_upd, 1e-12))
        total_ms = (time.perf_counter() - t0) * 1000.0
        rows.append({
            'dataset': DATASET_LABEL,
            'protocol': 'Kadem-style_GPR-EKF_Colab',
            'test_cell': row['cell'],
            'row_index': idx,
            'cycle_number': cycle,
            'measurement_source': f'Kadem-style GPR-EKF::{KADEM_PMAX_WINDOW}',
            'model': 'Kadem-style GPR-EKF',
            'window': KADEM_PMAX_WINDOW,
            'mode': mode,
            'availability': 1.0 if measurement_available else 0.0,
            'true_soh': y_true,
            'pred_soh': x_upd,
            'abs_error': abs(y_true - x_upd),
            'feature_assembly_time_ms': 0.0,
            'inference_time_ms': total_ms,
            'total_update_time_ms': total_ms,
            'model_size_kb': np.nan,
            'n_experts': 1,
            'measurement_val_rmse': np.nan,
        })
        x, P = x_upd, P_upd
        prev_cycle = cycle
    return pd.DataFrame(rows)

def evaluate_kadem_colab(dataframe):
    print('\n================ KADEM-STYLE GPR-EKF BASELINE ================')
    kdf = load_kadem_single_hi_table(dataframe, KADEM_PMAX_WINDOW)
    all_rows = []
    model_rows = []
    for test_cell in sorted(kdf['cell'].unique()):
        train_df = kdf[kdf['cell'] != test_cell].copy()
        test_df = kdf[kdf['cell'] == test_cell].copy()
        if len(train_df) < 20 or len(test_df) < 3:
            continue
        print('Kadem-style fold:', test_cell)
        pred_gpr, meas_gpr = fit_kadem_gpr_ekf_models(train_df)
        rows = run_kadem_gpr_ekf_on_cell(test_df, pred_gpr, meas_gpr, availability_prob=1.0, seed=RANDOM_STATE)
        size_kb = safe_model_size_kb({'pred_gpr': pred_gpr, 'meas_gpr': meas_gpr})
        rows['model_size_kb'] = size_kb
        all_rows.append(rows)
        model_rows.append({'dataset': DATASET_LABEL, 'test_cell': test_cell, 'model': 'Kadem-style GPR-EKF', 'window': KADEM_PMAX_WINDOW, 'model_size_kb': size_kb})
    return pd.concat(all_rows, ignore_index=True), pd.DataFrame(model_rows)

# ============================================================
# 7) Online replay and structured missing-window experiments
# ============================================================
def summarize_online(rows_df, group_cols=None):
    if group_cols is None:
        group_cols = ['dataset', 'protocol', 'measurement_source', 'model', 'window']
    out = []
    for keys, g in rows_df.groupby(group_cols, dropna=False):
        metrics = regression_metrics(g['true_soh'].values, g['pred_soh'].values)
        mode_counts = g['mode'].value_counts().to_dict()
        row = dict(zip(group_cols, keys))
        row.update(metrics)
        row.update({
            'n_rows': int(len(g)),
            'n_predictions': int(np.isfinite(g['pred_soh']).sum()),
            'measurement_update_count': int(mode_counts.get('measurement_update', 0)),
            'prediction_only_count': int(mode_counts.get('prediction_only', 0)),
            'no_measurement_count': int(mode_counts.get('no_measurement', 0)),
            'mean_availability': float(g['availability'].mean()),
            'mean_update_time_ms': float(g['total_update_time_ms'].mean()),
            'median_update_time_ms': float(g['total_update_time_ms'].median()),
            'p95_update_time_ms': float(g['total_update_time_ms'].quantile(0.95)),
            'mean_feature_assembly_time_ms': float(g['feature_assembly_time_ms'].mean()),
            'mean_inference_time_ms': float(g['inference_time_ms'].mean()),
            'mean_model_size_kb': float(g['model_size_kb'].mean()),
            'mean_n_experts': float(g['n_experts'].mean()),
            'measurement_val_rmse': float(g['measurement_val_rmse'].mean()),
        })
        out.append(row)
    return pd.DataFrame(out).sort_values(group_cols).reset_index(drop=True)

def evaluate_fast_runtime_ablation(dataframe):
    print('\n================ FAST MAW-GME RUNTIME ABLATION ================')
    cells = sorted(dataframe['cell'].unique())
    rows = []
    expert_rows = []
    ranking_rows = []
    for test_cell in cells:
        print('\nHeld-out cell:', test_cell)
        train_df = dataframe[dataframe['cell'] != test_cell].copy()
        test_df = dataframe[dataframe['cell'] == test_cell].copy().sort_values('cycle_number').reset_index(drop=True)
        if len(train_df) < 20 or len(test_df) < 3:
            print('Skipped due to insufficient train/test rows.')
            continue
        cand = inner_loco_rank_candidates(
            train_df,
            selected_models=MODEL_CANDIDATES,
            latency_aware=LATENCY_AWARE_PRUNING,
            size_penalty=SIZE_PENALTY,
        )
        cand_export = cand.drop(columns=['feature_cols'], errors='ignore').copy()
        cand_export['test_cell'] = test_cell
        cand_export['dataset'] = DATASET_LABEL
        ranking_rows.append(cand_export)
        best_direct = fit_direct_deploy(train_df, cand.iloc[0])
        full_maw = fit_maw_gme_deploy(train_df, cand, top_k=None, latency_label=False)
        deploys = [best_direct, full_maw]
        for k in TOP_KS:
            deploys.append(fit_maw_gme_deploy(train_df, cand, top_k=k, latency_label=LATENCY_AWARE_PRUNING))
        for deploy in deploys:
            measurement_source = deploy['kind']
            summary_model = deploy['model_name']
            summary_window = deploy['window']
            selected_model = deploy['model_name']
            selected_window = deploy['window']
            if deploy['kind'] == 'Best-single-estimator':
                measurement_source = 'Best-single-selected'
                summary_model = 'Best-single-selected'
                summary_window = 'per_fold_selected'
            if 'experts' in deploy:
                for e in deploy['experts']:
                    expert_rows.append({
                        'dataset': DATASET_LABEL,
                        'test_cell': test_cell,
                        'deployment_model': deploy['model_name'],
                        'expert_window': e['window'],
                        'expert_model': e['model_name'],
                        'expert_inner_rmse': e['inner_rmse'],
                        'expert_rank_score': e['rank_score'],
                        'expert_weight': e['base_weight_norm'],
                        'expert_model_size_kb': e['model_size_kb'],
                        'n_features': e['n_features'],
                    })
            last_pred = 1.0
            for idx, row in test_df.iterrows():
                true_soh = float(row['soh'])
                if 'experts' in deploy:
                    pred, availability, feature_ms, infer_ms, mode = predict_maw_one(deploy, row)
                else:
                    pred, availability, feature_ms, infer_ms, mode = predict_direct_one(deploy, row)
                if not np.isfinite(pred):
                    pred = last_pred
                    mode = 'prediction_only'
                else:
                    last_pred = pred
                rows.append({
                    'dataset': DATASET_LABEL,
                    'protocol': 'BMS-1_runtime_fast_pruned',
                    'test_cell': test_cell,
                    'row_index': idx,
                    'cycle_number': float(row['cycle_number']) if pd.notna(row['cycle_number']) else float(idx),
                    'measurement_source': measurement_source,
                    'model': summary_model,
                    'window': summary_window,
                    'selected_model': selected_model,
                    'selected_window': selected_window,
                    'mode': mode,
                    'availability': availability,
                    'true_soh': true_soh,
                    'pred_soh': pred,
                    'abs_error': abs(true_soh - pred),
                    'feature_assembly_time_ms': feature_ms,
                    'inference_time_ms': infer_ms,
                    'total_update_time_ms': feature_ms + infer_ms,
                    'model_size_kb': deploy['model_size_kb'],
                    'n_experts': deploy.get('n_experts', 1),
                    'measurement_val_rmse': deploy['val_rmse'],
                })
    rows_df = pd.DataFrame(rows)
    summary = summarize_online(rows_df)
    experts_df = pd.DataFrame(expert_rows)
    rankings_df = pd.concat(ranking_rows, ignore_index=True) if ranking_rows else pd.DataFrame()
    return rows_df, summary, experts_df, rankings_df

def is_pmax_feature_col(col):
    """Return True for Pmax-specific HI columns.

    This masks Pmax and normalized-Pmax measurements while keeping other
    IC/DVA-shape, throughput, time, temperature, and cycle features available.
    """
    c = str(col).lower()
    return ('pmax' in c) or ('p_max' in c)


def is_icdva_shape_feature_col(col):
    """Return True for IC/DVA curve-shape descriptors excluding Pmax.

    The aim is to create the BMS-3B feature-group stress scenario used in the
    manuscript: IC/DVA-shape features are unavailable, but Pmax, time,
    throughput, temperature, and cycle-number features remain observable.
    """
    c = str(col).lower()
    if is_pmax_feature_col(c):
        return False
    shape_tokens = [
        'ic_area', 'ic_mean', 'ic_std', 'ic_stddev', 'ic_skew', 'ic_kurt',
        'peak_width', 'width',
        'dva', 'dvdq', 'dv_dq', 'dv/dq',
        'dq_dv_shape', 'dqdv_shape'
    ]
    return any(tok in c for tok in shape_tokens)


def feature_group_mask_columns(columns, removed_feature_group):
    """Columns to mask for a feature-group stress scenario."""
    if removed_feature_group is None or str(removed_feature_group).lower() in {'', 'none'}:
        return []
    group = str(removed_feature_group).lower()
    if group == 'pmax':
        return [c for c in columns if is_pmax_feature_col(c)]
    if group in {'icdva_shape', 'ic_dva_shape', 'ic/dva_shape'}:
        return [c for c in columns if is_icdva_shape_feature_col(c)]
    raise ValueError(f'Unknown removed_feature_group={removed_feature_group}')


def apply_feature_group_mask(row, removed_feature_group):
    """Return a copy of row with selected feature group set to NaN.

    The model is not retrained for this BMS-3B stress test. Instead, the online
    measurement row is degraded to simulate the corresponding health-indicator
    family being unavailable at test time. The fitted imputer/model pipeline then
    handles the missing values, while the availability gate sees the reduced
    feature availability.
    """
    if removed_feature_group is None or str(removed_feature_group).lower() in {'', 'none'}:
        return row
    r = row.copy()
    masked_cols = feature_group_mask_columns(r.index, removed_feature_group)
    if masked_cols:
        r.loc[masked_cols] = np.nan
    return r


def build_structured_missing_scenarios(best_window):
    """Return the complete BMS-3B structured stress-test scenario set.

    The scenario set includes:
      1) consolidated deployment window cases,
      2) single-window-only diagnostic cases for every voltage window, and
      3) feature-group unavailability cases used in the ablation diagnostics
         (Pmax removed and IC/DVA-shape removed).

    Each scenario is represented as a dictionary so that both window availability
    and feature-group masking can be controlled by the same evaluation function.
    """
    all_windows = list(WINDOWS.keys())
    scenarios = [
        {
            'name': 'all_windows_available',
            'allowed_windows': all_windows,
            'removed_feature_group': None,
            'scenario_family': 'consolidated_window_availability',
        },
        {
            'name': 'best_window_only',
            'allowed_windows': [best_window],
            'removed_feature_group': None,
            'scenario_family': 'consolidated_window_availability',
        },
        {
            'name': 'best_window_unavailable',
            'allowed_windows': [w for w in all_windows if w != best_window],
            'removed_feature_group': None,
            'scenario_family': 'consolidated_window_availability',
        },
    ]
    # Individual single-window diagnostic scenarios, e.g. only_full_36_42,
    # only_medium_37_40, only_paper_335_350, only_very_narrow_335_340.
    for w in all_windows:
        scenarios.append({
            'name': f'only_{w}',
            'allowed_windows': [w],
            'removed_feature_group': None,
            'scenario_family': 'single_window_only_diagnostic',
        })
    # Feature-group unavailability diagnostics. All windows remain physically
    # observable, but the selected health-indicator family is masked at test time.
    scenarios.extend([
        {
            'name': 'pmax_features_removed',
            'allowed_windows': all_windows,
            'removed_feature_group': 'pmax',
            'scenario_family': 'feature_group_unavailability',
        },
        {
            'name': 'icdva_shape_features_removed',
            'allowed_windows': all_windows,
            'removed_feature_group': 'icdva_shape',
            'scenario_family': 'feature_group_unavailability',
        },
    ])
    return scenarios

def evaluate_structured_missing(dataframe):
    print('\n================ STRUCTURED MISSING-WINDOW TEST ================')
    cells = sorted(dataframe['cell'].unique())
    rows = []
    for test_cell in cells:
        print('\nHeld-out cell:', test_cell)
        train_df = dataframe[dataframe['cell'] != test_cell].copy()
        test_df = dataframe[dataframe['cell'] == test_cell].copy().sort_values('cycle_number').reset_index(drop=True)
        if len(train_df) < 20 or len(test_df) < 3:
            continue
        cand = inner_loco_rank_candidates(train_df, selected_models=MODEL_CANDIDATES, latency_aware=LATENCY_AWARE_PRUNING, size_penalty=SIZE_PENALTY)
        best_direct = fit_direct_deploy(train_df, cand.iloc[0])
        deploys = [best_direct]
        for k in TOP_KS:
            deploys.append(fit_maw_gme_deploy(train_df, cand, top_k=k, latency_label=LATENCY_AWARE_PRUNING))
        scenarios = build_structured_missing_scenarios(best_direct['window'])
        for scenario in scenarios:
            scenario_name = scenario['name']
            allowed_windows = scenario['allowed_windows']
            removed_feature_group = scenario.get('removed_feature_group', None)
            scenario_family = scenario.get('scenario_family', 'unspecified')
            allowed_set = set(allowed_windows)
            for deploy in deploys:
                measurement_source = deploy['kind']
                summary_model = deploy['model_name']
                summary_window = deploy['window']
                selected_model = deploy['model_name']
                selected_window = deploy['window']
                if deploy['kind'] == 'Best-single-estimator':
                    measurement_source = 'Best-single-selected'
                    summary_model = 'Best-single-selected'
                    summary_window = 'per_fold_selected'
                last_pred = 1.0
                for idx, row in test_df.iterrows():
                    true_soh = float(row['soh'])
                    stress_row = apply_feature_group_mask(row, removed_feature_group)
                    if 'experts' in deploy:
                        pred, availability, feature_ms, infer_ms, mode = predict_maw_one(deploy, stress_row, allowed_windows=allowed_set)
                    else:
                        if deploy['window'] not in allowed_set:
                            pred, availability, feature_ms, infer_ms, mode = np.nan, 0.0, 0.0, 0.0, 'no_measurement'
                        else:
                            pred, availability, feature_ms, infer_ms, mode = predict_direct_one(deploy, stress_row)
                    if not np.isfinite(pred):
                        pred = last_pred
                        mode = 'prediction_only'
                    else:
                        last_pred = pred
                    rows.append({
                        'dataset': DATASET_LABEL,
                        'protocol': 'BMS-3B_structured_missing_fast_pruned',
                        'test_cell': test_cell,
                        'row_index': idx,
                        'cycle_number': float(row['cycle_number']) if pd.notna(row['cycle_number']) else float(idx),
                        'stress_scenario': scenario_name,
                        'scenario_family': scenario_family,
                        'allowed_windows': ';'.join(allowed_windows),
                        'removed_feature_group': removed_feature_group if removed_feature_group is not None else 'none',
                        'measurement_source': measurement_source,
                        'model': summary_model,
                        'window': summary_window,
                        'selected_model': selected_model,
                        'selected_window': selected_window,
                        'mode': mode,
                        'availability': availability,
                        'true_soh': true_soh,
                        'pred_soh': pred,
                        'abs_error': abs(true_soh - pred),
                        'feature_assembly_time_ms': feature_ms,
                        'inference_time_ms': infer_ms,
                        'total_update_time_ms': feature_ms + infer_ms,
                        'model_size_kb': deploy['model_size_kb'],
                        'n_experts': deploy.get('n_experts', 1),
                        'measurement_val_rmse': deploy['val_rmse'],
                    })
    rows_df = pd.DataFrame(rows)
    summary = summarize_online(rows_df, group_cols=['dataset','protocol','stress_scenario','measurement_source','model','window'])
    return rows_df, summary

# ============================================================
# 8) RUN ALL COMPARISONS
# ============================================================
np.random.seed(RANDOM_STATE)

# Kadem-style single-HI GPR-EKF in the same Colab/sklearn environment.
kadem_rows, kadem_model_sizes = evaluate_kadem_colab(df)
kadem_summary = summarize_online(kadem_rows)
kadem_summary['paper_reported_gpr_ekf_rmse_context'] = PAPER_GPR_EKF_RMSE
kadem_summary['paper_reported_gpr_ekf_mae_context'] = PAPER_GPR_EKF_MAE

# Fast/pruned MAW-GME ablation.
fast_rows, fast_summary, fast_experts, candidate_rankings = evaluate_fast_runtime_ablation(df)

# Structured missing-window robustness for pruned versions.
missing_rows, missing_summary = evaluate_structured_missing(df)

# Optional: load original unoptimized BMS-1 summary from the previous notebook, if present.
def load_original_bms1_summary_if_available():
    old_path = os.path.join(RESULT_DIR, 'bms_online_results', 'bms1_online_replay_summary.csv')
    if not os.path.exists(old_path):
        print('Original BMS-1 summary not found; skipping:', old_path)
        return pd.DataFrame()
    old = pd.read_csv(old_path).copy()
    old['protocol'] = 'BMS-1_original_unoptimized_summary'
    rename_map = {
        'RMSE': 'rmse',
        'MAE': 'mae',
        'R2': 'r2',
        'mean_total_update_time_ms': 'mean_update_time_ms',
        'median_total_update_time_ms': 'median_update_time_ms',
    }
    old = old.rename(columns=rename_map)
    if 'p95_update_time_ms' not in old.columns:
        old['p95_update_time_ms'] = np.nan
    if 'mean_feature_assembly_time_ms' not in old.columns:
        old['mean_feature_assembly_time_ms'] = np.nan
    if 'mean_inference_time_ms' not in old.columns:
        old['mean_inference_time_ms'] = np.nan
    if 'n_eval' not in old.columns:
        old['n_eval'] = old.get('n_updates', old.get('n_rows', np.nan))
    print('Loaded original BMS-1 summary:', old_path)
    return old

original_bms1_summary = load_original_bms1_summary_if_available()

# Combined runtime table for manuscript Table 6 revision.
combined_runtime_summary = pd.concat([original_bms1_summary, kadem_summary, fast_summary], ignore_index=True, sort=False)

print('\n================ KADEM-STYLE SUMMARY ================')
display(kadem_summary)
print('\n================ FAST RUNTIME SUMMARY ================')
display(fast_summary)
print('\n================ COMBINED RUNTIME SUMMARY ================')
display(combined_runtime_summary)
print('\n================ STRUCTURED MISSING SUMMARY ================')
display(missing_summary)
print('\n================ SELECTED EXPERTS ================')
display(fast_experts.head(50))

# ============================================================
# 9) SAVE OUTPUTS
# ============================================================
paths = {
    'kadem_rows': os.path.join(FAST_OUT_DIR, 'kadem_style_colab_online_rows.csv'),
    'kadem_summary': os.path.join(FAST_OUT_DIR, 'kadem_style_colab_summary.csv'),
    'kadem_model_sizes': os.path.join(FAST_OUT_DIR, 'kadem_style_colab_model_sizes.csv'),
    'fast_rows': os.path.join(FAST_OUT_DIR, 'bms1_full_model_pool_runtime_rows.csv'),
    'fast_summary': os.path.join(FAST_OUT_DIR, 'bms1_full_model_pool_runtime_summary.csv'),
    'fast_experts': os.path.join(FAST_OUT_DIR, 'bms1_full_model_pool_selected_experts.csv'),
    'candidate_rankings': os.path.join(FAST_OUT_DIR, 'candidate_expert_rankings_by_outer_fold.csv'),
    'missing_rows': os.path.join(FAST_OUT_DIR, 'bms3b_full_model_pool_structured_missing_rows.csv'),
    'missing_summary': os.path.join(FAST_OUT_DIR, 'bms3b_full_model_pool_structured_missing_summary.csv'),
    'original_bms1_summary': os.path.join(FAST_OUT_DIR, 'original_bms1_summary_loaded.csv'),
    'combined_runtime_summary': os.path.join(FAST_OUT_DIR, 'manuscript_table6_full_model_pool_runtime_combined.csv'),
    'excel': os.path.join(FAST_OUT_DIR, 'full_model_pool_runtime_review_outputs.xlsx'),
}

kadem_rows.to_csv(paths['kadem_rows'], index=False)
kadem_summary.to_csv(paths['kadem_summary'], index=False)
kadem_model_sizes.to_csv(paths['kadem_model_sizes'], index=False)
fast_rows.to_csv(paths['fast_rows'], index=False)
fast_summary.to_csv(paths['fast_summary'], index=False)
fast_experts.to_csv(paths['fast_experts'], index=False)
candidate_rankings.to_csv(paths['candidate_rankings'], index=False)
missing_rows.to_csv(paths['missing_rows'], index=False)
missing_summary.to_csv(paths['missing_summary'], index=False)
original_bms1_summary.to_csv(paths['original_bms1_summary'], index=False)
combined_runtime_summary.to_csv(paths['combined_runtime_summary'], index=False)

with pd.ExcelWriter(paths['excel']) as writer:
    combined_runtime_summary.to_excel(writer, sheet_name='table6_runtime_combined', index=False)
    original_bms1_summary.to_excel(writer, sheet_name='original_bms1_summary', index=False)
    kadem_summary.to_excel(writer, sheet_name='kadem_summary', index=False)
    fast_summary.to_excel(writer, sheet_name='fast_runtime_summary', index=False)
    missing_summary.to_excel(writer, sheet_name='structured_missing_summary', index=False)
    fast_experts.to_excel(writer, sheet_name='selected_experts', index=False)
    candidate_rankings.to_excel(writer, sheet_name='candidate_rankings', index=False)

print('\nSaved outputs:')
for k, v in paths.items():
    print(k, '->', v)

# Compact table to copy into manuscript/reviewer response.
copy_cols = [
    'dataset','measurement_source','model','window','rmse','mae','n_eval',
    'mean_update_time_ms','median_update_time_ms','p95_update_time_ms',
    'mean_model_size_kb','mean_n_experts'
]
compact = combined_runtime_summary[[c for c in copy_cols if c in combined_runtime_summary.columns]].copy()
print('\nCompact Table 6 candidate:')
display(compact)
